<a href="https://colab.research.google.com/github/Nasim-Amani/tag-recommender-agent-learning/blob/main/L1_Baselines_Popularity_%2B_text_similarity_(BERT).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning plan: from simple baselines to a tagging agent

**Idea:** learn by doing. One dataset, one results table, **one new idea per level**. Each level adds a new row to the table, so I can see with my own eyes whether the idea helped.

**Fixed for every level:**
- **Data:** Amazon Reviews 2023, All_Beauty (first 2,000 products; tags = seller attributes)
- **Split:** by first-review date: oldest 70% train, next 15% validation, newest 15% test (= new products)
- **Results table:** Precision@10, Recall@10, NDCG@10 on the test products

| Level | New idea | What I build | Read first |
|---|---|---|---|
| **1** | Baselines | Random, Popularity, Text similarity (BERT) → first rows of the table | nothing new |
| **2** | Borrow from neighbors | tags of my 5 most similar products | nothing new |
| **3** | Graph | LightGCN on a product–tag graph | LightGCN with PyTorch Geometric (Stanford CS224W) |
| **4** | Hypergraph *(optional)* | HGNN with phrase nodes | DHG library tutorial, HGNN+ example |
| **5** | LLM tagging | give an LLM the product text + tag list, ask for top-10 tags; then add the neighbors' tags to the prompt | nothing (just an API call) |
| **6** | Agent | the LLM uses **tools** (find similar products, check the tag list, check the reviews), then decides | LangGraph quickstart |

**Why this order:**
- **Levels 1–2:** set the bar to beat. Anything complex must beat "copy the neighbors' tags".
- **Levels 3–4:** show what graphs add, so I understand CASTRA and can judge it fairly.
- **Levels 5–6:** my proposal: an LLM reads the product (syntax, sentiment, meaning) and an agent gathers the neighbors' tags by itself ("syntactic patterns + neighboring tag attributes").

**Two rules:**
1. Learn a new word only when a level needs it.
2. Every level ends with the table: if I can't explain why a row went up or down, I stay on that level.

# Level 1: Baselines

**Goal:** before building anything smart, measure how well **simple methods** do. This sets the bar every later level must beat.

**Three simple methods:**
- **Random:** shuffle the tags (the worst possible; any real method must beat it).
- **Popularity:** give every product the most common tags (no learning, no reading).
- **Text similarity:** compare the product's text with each tag's text using BERT vectors (reads the product, but no training).

**The test is fair:** we split products by **first-review date**. The newest 15% are the **test products**; they play the role of new products whose tags we must guess.

In [1]:
import json, gzip, os, urllib.request
from collections import Counter
import numpy as np
import pandas as pd

# 1) Download the two files (only the first time)
base = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
for url, name in [(f"{base}/meta_categories/meta_All_Beauty.jsonl.gz", "meta.jsonl.gz"),
                  (f"{base}/review_categories/All_Beauty.jsonl.gz", "reviews.jsonl.gz")]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(url, name)

# 2) Read the first 2,000 products
products = []
for line in gzip.open("meta.jsonl.gz", "rt"):
    products.append(json.loads(line))
    if len(products) == 2000:
        break

# 3) Reviews and first-review time of those products
reviews = {p["parent_asin"]: [] for p in products}
first_time = {}
for line in gzip.open("reviews.jsonl.gz", "rt"):
    r = json.loads(line)
    a = r["parent_asin"]
    if a in reviews:
        reviews[a].append(r["text"])
        first_time[a] = min(first_time.get(a, r["timestamp"]), r["timestamp"])

# 4) Tags = values of seller attributes, minus identifiers and junk keys
SKIP = {"Brand", "Manufacturer", "UPC", "Product Dimensions", "Item model number",
        "Date First Available", "Item Weight", "Package Dimensions",
        "Is Discontinued By Manufacturer", "Unit Count", "Number of Items",
        "Age Range (Description)", "Mounting Type", "Installation Type",
        "Package Information", "Extension Length"}

def all_tags_of(p):
    tags = []
    for key, value in p["details"].items():
        if key in SKIP or not isinstance(value, str):
            continue
        for v in value.split(","):                 # "Oily, Dry" -> two tags
            v = v.strip().lower()
            if v and v != "all":                   # "suits everyone" is not a tag
                tags.append(f"{key}: {v}")
    return tags

counts = Counter(t for p in products for t in all_tags_of(p))
candidate_tags = sorted(t for t, c in counts.items() if c >= 20)   # keep tags on >= 20 products

# Keep only products with at least one candidate tag (otherwise there is nothing to evaluate)
products = [p for p in products if set(all_tags_of(p)) & set(candidate_tags)]
n, m = len(products), len(candidate_tags)

# 5) Answer key Y: Y[k, j] = 1 if product k has tag j
Y = np.array([[t in all_tags_of(p) for t in candidate_tags] for p in products], dtype=float)

# 6) Split by time: oldest 70% train, next 15% validation, newest 15% test (= "new products")
t = np.array([first_time.get(p["parent_asin"], np.inf) for p in products])   # no reviews -> newest
order = np.argsort(t)
train_idx = order[: int(0.70 * n)]
val_idx   = order[int(0.70 * n): int(0.85 * n)]
test_idx  = order[int(0.85 * n):]

print(n, "products |", m, "tags |", len(train_idx), "train /", len(val_idx), "val /", len(test_idx), "test")
print("average tags per product:", round(Y.sum(axis=1).mean(), 2))

1057 products | 35 tags | 739 train / 159 val / 159 test
average tags per product: 1.75


## How we measure

For each test product, a method gives every tag a **score**; we keep the **top 10**, and compare them with the true tags:
- **Precision@10:** how many of the 10 are correct (÷ 10)
- **Recall@10:** how many of the true tags we found (÷ number of true tags)
- **NDCG@10:** like Recall, but a correct tag at position 1 counts more than at position 10 (1 = perfect order)

All range from 0 to 1; higher is better.

In [2]:
def evaluate(scores, products_idx, K=10):
    """scores: (number of products, number of tags), higher = more likely.
    Returns Precision@K, Recall@K and NDCG@K, averaged over the products."""
    truth = Y[products_idx]                                     # true tags of these products
    top = np.argsort(-scores, axis=1)[:, :K]                    # the K best tags per product
    hits = np.take_along_axis(truth, top, axis=1)               # 1 where a recommended tag is correct
    n_true = truth.sum(axis=1)                                  # how many true tags each product has
    discount = 1 / np.log2(np.arange(2, K + 2))                 # position 1 counts most, position 10 least
    ideal = np.array([discount[:int(min(nt, K))].sum() for nt in n_true])   # best possible score
    return {"Precision@10": (hits.sum(axis=1) / K).mean(),
            "Recall@10":    (hits.sum(axis=1) / n_true).mean(),
            "NDCG@10":      ((hits * discount).sum(axis=1) / ideal).mean()}

In [3]:
results = {}

# Random: random scores
rng = np.random.default_rng(0)
results["Random"] = evaluate(rng.random((len(test_idx), m)), test_idx)

# Popularity: count how many TRAIN products have each tag; same list for every product
tag_popularity = Y[train_idx].sum(axis=0)
results["Popularity"] = evaluate(np.tile(tag_popularity, (len(test_idx), 1)), test_idx)

# Text similarity: cosine similarity between product text and tag text (BERT vectors)
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("all-MiniLM-L6-v2")
texts = [p["title"] + " " + " ".join(p["features"] + p["description"]) for p in products]
Z_prod = encoder.encode(texts, normalize_embeddings=True)          # (n, 384) product meaning
Z_tag = encoder.encode(candidate_tags, normalize_embeddings=True)  # (m, 384) tag meaning
results["Text similarity"] = evaluate(Z_prod[test_idx] @ Z_tag.T, test_idx)

table = pd.DataFrame(results).T.round(3)
table

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,Precision@10,Recall@10,NDCG@10
Random,0.048,0.241,0.133
Popularity,0.116,0.573,0.340
Text similarity,0.123,0.651,0.484




**Goal:** see how well simple methods guess the tags of **new** (test) products, to set the bar for later levels.

| Method | Precision@10 | Recall@10 | NDCG@10 | In simple words |
|---|---|---|---|---|
| Random | 0.048 | 0.241 | 0.133 | guessing blindly finds only **1 in 4** true tags |
| Popularity | 0.116 | 0.573 | 0.340 | just recommending common tags already finds **more than half** |
| **Text similarity** | **0.123** | **0.651** | **0.484** | reading the product finds **2 in 3** true tags, and puts them **higher** in the list |

**What this means:**
- **Both real methods beat Random**, so the setup works.
- **Popularity is surprisingly strong:** a few common tags cover many products. This is the same effect we saw in the CASTRA paper.
- **Text similarity wins on all three numbers**, especially **NDCG (0.48 vs 0.34)**: it doesn't just find more true tags, it also places them **near the top** of the list.
- **Precision looks small (~0.12)** because most products have only ~2–3 true tags, so most of the 10 slots are necessarily wrong. That's normal.

**Bar for Level 2:** a new method must beat **NDCG@10 = 0.484**.

# **My Ambiguities**

## The 3 evaluation metrics, simply

**Setup:** for each product, the method gives a **top-10 list** of tags. We compare it with the product's **true tags**.

**Example:** a shampoo with **3 true tags**: `dry hair`, `argan oil`, `liquid`.
The method's top-10 list has `dry hair` at **position 1**, `liquid` at **position 4**, and 8 wrong tags. It missed `argan oil`.

| Metric | The question it answers | Example |
|---|---|---|
| **Precision@10** | Of the 10 tags I recommended, how many are **correct**? | 2 ÷ 10 = **0.20** |
| **Recall@10** | Of the true tags, how many did I **find**? | 2 ÷ 3 = **0.67** |
| **NDCG@10** | Did I put the correct tags **near the top**? | **0.67** (see below) |

**NDCG in plain words:** each correct tag earns points, **more points if it's higher in the list**:
- position 1 → 1.00 point
- position 2 → 0.63
- position 3 → 0.50
- position 4 → 0.43 … position 10 → 0.29

Our list: 1.00 (position 1) + 0.43 (position 4) = **1.43**.
The **perfect** list (all 3 true tags at positions 1, 2, 3): 1.00 + 0.63 + 0.50 = **2.13**.
NDCG = 1.43 ÷ 2.13 = **0.67**, i.e. "67% as good as the perfect order".

**How to read the Level 1 table** (Text similarity row):
- **Recall 0.65:** on average, **65%** of a product's true tags appear in its top 10.
- **Precision 0.12:** on average, about **1 of the 10** recommended tags is correct.
- **NDCG 0.48:** the order is about **half as good as perfect**.

**Why Precision is always small here:** most products have only ~2–3 true tags, so even a perfect list of 10 has at most 2–3 correct. Precision can't go much above **0.3**. Recall and NDCG are the more useful numbers.

**All three:** range 0 to 1, higher is better.

### How text similarity (BERT) recommends tags

**Key point:** the method compares each product with **all candidate tags**, never with its true tags. True tags are used **only at the end**, to check the answer.

**Steps (example: "Argan Oil Repair Shampoo: restores damaged, frizzy hair")**
1. **Product → vector:** BERT turns the product's text into 384 numbers (its meaning).
2. **Every tag → vector:** BERT turns each tag name ("Hair Type: dry", "Scent: lavender", …) into 384 numbers.
3. **Compare:** similarity between the product and every tag (0 = unrelated, 1 = same meaning):
   `Hair Type: dry` 0.52 · `Item Form: liquid` 0.41 · `Scent: lavender` 0.15 · `Item Form: powder` 0.09 …
4. **Top 10:** sort by similarity and keep the 10 best tags.
5. **Check:** `evaluate` compares these 10 with the true tags.

**vs Popularity:** Popularity scores tags by **how common** they are (same list for everyone); text similarity scores them by **how close they are in meaning to this product** (a different list for each product).

In code: `scores = Z_prod[test_idx] @ Z_tag.T` → (test products × all tags)